In [ ]:
import time

start = time.time()

def time_check():
    end = time.time()
    
    # Time in minutes
    print(f"Run time: {round((end - start)/60, 2)} minutes")

In [ ]:
import pandas as pd
import sys

test = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv")

if len(test) == 200:
    test['prediction'] = 'A B C'

    submission = test[['id', 'prediction']]
    submission.to_csv('submission.csv', index=False)
    
    sys.exit()

# Installations

In [ ]:
!cp /kaggle/input/datasets-wheel/datasets-2.14.4-py3-none-any.whl /kaggle/working
!pip install  /kaggle/working/datasets-2.14.4-py3-none-any.whl
# !cp /kaggle/input/backup-806/util_openbook.py .

In [ ]:
# installing offline dependencies
!pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

!pip install --no-index --no-deps /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

In [ ]:
import ctypes
import gc
import pickle
import torch

libc = ctypes.CDLL("libc.so.6")

In [ ]:
time_check()

# Orig Wiki Context

In [ ]:
from util_origwikicontext import get_contexts

get_contexts()

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

In [ ]:
time_check()

# Read data

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd 
from datasets import load_dataset, load_from_disk
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from transformers import LongformerTokenizer, LongformerForMultipleChoice
import transformers
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import unicodedata

from transformers import AutoTokenizer
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from torch.utils.data import DataLoader

import os

In [ ]:
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from scipy.special import softmax

In [ ]:
DEBUG = False

if DEBUG: 
    df_valid1 = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv")
    df_valid2 = pd.read_csv("/kaggle/input/llmse-eval/LLMSE_1k.csv").sample(600)
    
    df_valid = pd.concat([df_valid1, df_valid2], ignore_index=True)
    df_valid = df_valid.reset_index(drop=True)
    
#     df_valid = pd.concat([df_valid] * 5, ignore_index=True)
    
    del df_valid1, df_valid2
    _ = gc.collect()
    
else:
    df_valid = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv")
    
df_valid.shape

In [ ]:
# if len(df_valid) == 200: df_valid = df_valid.head()

In [ ]:
stop_words = ['each', 'you', 'the', 'use', 'used',
                  'where', 'themselves', 'nor', "it's", 'how', "don't", 'just', 'your',
                  'about', 'himself', 'with', "weren't", 'hers', "wouldn't", 'more', 'its', 'were',
                  'his', 'their', 'then', 'been', 'myself', 're', 'not',
                  'ours', 'will', 'needn', 'which', 'here', 'hadn', 'it', 'our', 'there', 'than',
                  'most', "couldn't", 'both', 'some', 'for', 'up', 'couldn', "that'll",
                  "she's", 'over', 'this', 'now', 'until', 'these', 'few', 'haven',
                  'of', 'wouldn', 'into', 'too', 'to', 'very', 'shan', 'before', 'the', 'they',
                  'between', "doesn't", 'are', 'was', 'out', 'we', 'me',
                  'after', 'has', "isn't", 'have', 'such', 'should', 'yourselves', 'or', 'during', 'herself',
                  'doing', 'in', "shouldn't", "won't", 'when', 'do', 'through', 'she',
                  'having', 'him', "haven't", 'against', 'itself', 'that',
                  'did', 'theirs', 'can', 'those',
                  'own', 'so', 'and', 'who', "you've", 'yourself', 'her', 'he', 'only',
                  'what', 'ourselves', 'again', 'had', "you'd", 'is', 'other',
                  'why', 'while', 'from', 'them', 'if', 'above', 'does', 'whom',
                  'yours', 'but', 'being', "wasn't", 'be']

In [ ]:
!cp -r /kaggle/input/stem-wiki-cohere-no-emb /kaggle/working
!cp -r /kaggle/input/all-paraphs-parsed-expanded /kaggle/working/

In [ ]:
time_check()

# Custom functions for retrieval

In [ ]:
def SplitList(mylist, chunk_size):
    return [mylist[offs:offs+chunk_size] for offs in range(0, len(mylist), chunk_size)]

def get_relevant_documents_parsed(df_valid):
    df_chunk_size=600
    paraphs_parsed_dataset = load_from_disk("/kaggle/working/all-paraphs-parsed-expanded")
    modified_texts = paraphs_parsed_dataset.map(lambda example:
                                             {'temp_text':
                                              f"{example['title']} {example['section']} {example['text']}".replace('\n'," ").replace("'","")},
                                             num_proc=2)["temp_text"]
    
    if len(df_valid) == 5: modified_texts = modified_texts[:10000]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         paraphs_parsed_dataset[idx.item()]["title"],
                         paraphs_parsed_dataset[idx.item()]["text"],
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    
    del paraphs_parsed_dataset, modified_texts, articles_flatten, articles_values_array, article_indices_array
    _ = gc.collect()
    
    return retrieved_articles



def get_relevant_documents(df_valid):
    df_chunk_size=800
    
    cohere_dataset_filtered = load_from_disk("/kaggle/working/stem-wiki-cohere-no-emb")
    modified_texts = cohere_dataset_filtered.map(lambda example:
                                             {'temp_text':
                                              unicodedata.normalize("NFKD", f"{example['title']} {example['text']}").replace('"',"")},
                                             num_proc=2)["temp_text"]
    
    if len(df_valid) == 5: modified_texts = modified_texts[:10000]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         cohere_dataset_filtered[idx.item()]["title"],
                         unicodedata.normalize("NFKD", cohere_dataset_filtered[idx.item()]["text"]),
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    
    del cohere_dataset_filtered, modified_texts, articles_flatten, articles_values_array, article_indices_array
    _ = gc.collect()
    
    return retrieved_articles



def retrieval(df_valid, modified_texts):
    
    corpus_df_valid = df_valid.apply(lambda row:
                                     f'{row["prompt"]}\n{row["prompt"]}\n{row["prompt"]}\n{row["A"]}\n{row["B"]}\n{row["C"]}\n{row["D"]}\n{row["E"]}',
                                     axis=1).values
    vectorizer1 = TfidfVectorizer(ngram_range=(1,2),
                                 token_pattern=r"(?u)\b[\w/.-]+\b|!|/|\?|\"|\'",
                                 stop_words=stop_words)
    vectorizer1.fit(corpus_df_valid)
    vocab_df_valid = vectorizer1.get_feature_names_out()
    vectorizer = TfidfVectorizer(ngram_range=(1,2),
                                 token_pattern=r"(?u)\b[\w/.-]+\b|!|/|\?|\"|\'",
                                 stop_words=stop_words,
                                 vocabulary=vocab_df_valid)
    vectorizer.fit(modified_texts[:500000])
    corpus_tf_idf = vectorizer.transform(corpus_df_valid)
    
    print(f"length of vectorizer vocab is {len(vectorizer.get_feature_names_out())}")

    chunk_size = 100000
    top_per_chunk = 10
    top_per_query = 10

    all_chunk_top_indices = []
    all_chunk_top_values = []

    for idx in tqdm(range(0, len(modified_texts), chunk_size)):
        wiki_vectors = vectorizer.transform(modified_texts[idx: idx+chunk_size])
        temp_scores = (corpus_tf_idf * wiki_vectors.T).toarray()
        chunk_top_indices = temp_scores.argpartition(-top_per_chunk, axis=1)[:, -top_per_chunk:]
        chunk_top_values = temp_scores[np.arange(temp_scores.shape[0])[:, np.newaxis], chunk_top_indices]

        all_chunk_top_indices.append(chunk_top_indices + idx)
        all_chunk_top_values.append(chunk_top_values)

    top_indices_array = np.concatenate(all_chunk_top_indices, axis=1)
    top_values_array = np.concatenate(all_chunk_top_values, axis=1)
    
    merged_top_scores = np.sort(top_values_array, axis=1)[:,-top_per_query:]
    merged_top_indices = top_values_array.argsort(axis=1)[:,-top_per_query:]
    articles_indices = top_indices_array[np.arange(top_indices_array.shape[0])[:, np.newaxis], merged_top_indices]
    
    return articles_indices, merged_top_scores


def prepare_answering_input(
        tokenizer, 
        question,  
        options,   
        context,   
        max_seq_length=4096,
    ):
    c_plus_q   = context + ' ' + tokenizer.bos_token + ' ' + question
    c_plus_q_4 = [c_plus_q] * len(options)
    tokenized_examples = tokenizer(
        c_plus_q_4, options,
        max_length=max_seq_length,
        padding="longest",
        truncation=False,
        return_tensors="pt",
    )
    input_ids = tokenized_examples['input_ids'].unsqueeze(0)
    attention_mask = tokenized_examples['attention_mask'].unsqueeze(0)
    example_encoded = {
        "input_ids": input_ids.to(model.device.index),
        "attention_mask": attention_mask.to(model.device.index),
    }
    return example_encoded


In [ ]:
time_check()

# Retrieval 1

In [ ]:
retrieved_articles_parsed = get_relevant_documents_parsed(df_valid)

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

In [ ]:
contexts = []

for index in tqdm(range(0,len(df_valid))):   
    
    context = f"{retrieved_articles_parsed[index][-3][2]}\n{retrieved_articles_parsed[index][-2][2]}\n{retrieved_articles_parsed[index][-1][2]}"
    contexts.append(context)

In [ ]:
df_valid['context'] = contexts

df_valid[["id", "prompt", "A", "B", "C", "D", "E", "context"]].to_csv("./test_context2.csv", index=False)

In [ ]:
del retrieved_articles_parsed

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

In [ ]:
time_check()

# Retrieval 2

In [ ]:
retrieved_articles = get_relevant_documents(df_valid)

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

In [ ]:
contexts = []

for index in tqdm(range(0,len(df_valid))):
    
    context = f"{retrieved_articles[index][-4][2]}\n{retrieved_articles[index][-3][2]}\n{retrieved_articles[index][-2][2]}\n{retrieved_articles[index][-1][2]}"
    contexts.append(context)

In [ ]:
df_valid['context'] = contexts

df_valid[["id", "prompt", "A", "B", "C", "D", "E", "context"]].to_csv("./test_context3.csv", index=False)

In [ ]:
del retrieved_articles

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

In [ ]:
time_check()

# Find Best Context

In [ ]:
test1 = pd.read_csv("test_context1.csv")
test1 = test1[['id','context']].copy()
test1.rename(columns={'context': 'context1'}, inplace=True)

test2 = pd.read_csv("test_context2.csv")
test2.rename(columns={'context': 'context2'}, inplace=True)

test3 = pd.read_csv("test_context3.csv")
test3 = test3[['id','context']].copy()
test3.rename(columns={'context': 'context3'}, inplace=True)

test = pd.merge(test2, test1, on="id", how="left")
test = pd.merge(test, test3, on="id", how="left")

test = test[[col for col in test.columns if 'context' not in col] + ['context1','context2','context3']]

print(test.shape)
test.head(2)

In [ ]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
embed_model = "/kaggle/input/bge-large-en-v1-5/bge-large-en-v1.5"
model = SentenceTransformer(embed_model)

def compute_relevance(prompt, contexts):
    
    prompt_embedding = model.encode(prompt)
    
    similarities = {}
    for idx, context in enumerate(contexts, 1):
        context_embedding = model.encode(context)
        # Use 1 - cosine() since we want similarity, not distance
        similarities[f"context{idx}"] = 1 - cosine(prompt_embedding, context_embedding)
    
    return similarities

In [ ]:
%%capture

final_context = []

for row in test.itertuples():
    
    prompt = row.prompt
    contexts = [row.context1,row.context2,row.context3]

    # Get relevance scores
    relevances = compute_relevance(prompt, contexts)
    
    max_context = max(relevances, key=relevances.get)
    
    final_context.append(getattr(row, max_context))


test['context'] = final_context 



In [ ]:
del model

_ = gc.collect()
libc.malloc_trim(0)
torch.cuda.empty_cache()

In [ ]:
test[["id", "prompt", "A", "B", "C", "D", "E", "context"]].to_csv("./test_context.csv", index=False)

# Ensemble definition

In [ ]:
model_dir_Lst = [
            "/kaggle/input/llmse-illift2/Debertav3Large-32k-256-r2-checkpoint-250-20230928T200331Z-002/checkpoint-250-80/",
            "/kaggle/input/llmse-illift-1kvals/Longformer-checkpoint-1850-20231004T033256Z-001/Longformer-checkpoint-1850/",
            "/kaggle/input/llmse-illift-1kvals/Deberta-PreprocMod-32k-384-checkpoint-950-753-20231004T041441Z-001/Deberta-PreprocMod-32k-384-checkpoint-950-753/",
            "/kaggle/input/llmse-illift-1kvals/Longformer-checkpoint-725-20231004T033922Z-001/Longformer-checkpoint-725/",
    

           ]

wts = [0.57976125, 0.2915075, 0.08835625, 0.040375]

sum(wts)

In [ ]:
options = 'ABCDE'
indices = list(range(5))

option_to_index = {option: index for option, index in zip(options, indices)}
index_to_option = {index: option for option, index in zip(options, indices)}

def preprocess(example):
  
    first_sentence = [example['prompt']] * 5
    second_sentence = []
    for option in options:
        second_sentence.append(example[option])
    
    tokenized_example = tokenizer(first_sentence, second_sentence, truncation='only_first')
    tokenized_example['label'] = option_to_index[example['answer']]
    return tokenized_example


def preprocess_longformer(example):
    first_sentence = [ "[CLS] " + example['prompt'] ] * 5
    # second_sentences = [" #### " + example['prompt'] + " [SEP] " + example[option] + " [SEP]" for option in 'ABCDE']
    second_sentences = [" #### <s>" + " </s></s> " + example[option] + " </s>" for option in 'ABCDE']
    tokenized_example = tokenizer(first_sentence, second_sentences, truncation='only_first',
                                  add_special_tokens=False)
#     max_length=MAX_INPUT, 
    tokenized_example['label'] = option_to_index[example['answer']]

    return tokenized_example

In [ ]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    
    def __call__(self, features):
        label_name = "label" if 'label' in features[0].keys() else 'labels'
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]['input_ids'])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt',
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch['labels'] = torch.tensor(labels, dtype=torch.int64)
        return batch

# Inference

In [ ]:
CONTEXT_LEN = 4096

test_df = pd.read_csv("test_context.csv")
test_df.index = list(range(len(test_df)))
test_df['id'] = list(range(len(test_df)))
test_df["prompt"] = test_df["context"].apply(lambda x: x[:CONTEXT_LEN]) + " #### " +  test_df["prompt"]
test_df['answer'] = 'A'

for i, model_dir in enumerate(model_dir_Lst):
    
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForMultipleChoice.from_pretrained(model_dir).cuda()
    
    if 'Longformer' in model_dir:
        tokenized_test_dataset = Dataset.from_pandas(test_df[['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']].drop(columns=['id'])).map(preprocess_longformer, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
    else:
        tokenized_test_dataset = Dataset.from_pandas(test_df[['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']].drop(columns=['id'])).map(preprocess, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
    
    tokenized_test_dataset = tokenized_test_dataset.remove_columns(["__index_level_0__"])
    data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)
    test_dataloader = DataLoader(tokenized_test_dataset, batch_size=1, shuffle=False, collate_fn=data_collator)
 
    
    globals()[f'predictions_model{i}'] = []
    
    for batch in tqdm(test_dataloader):
        for k in batch.keys():
            batch[k] = batch[k].cuda()
        with torch.no_grad():
            outputs = model(**batch)
        globals()[f'predictions_model{i}'].append(outputs.logits.cpu().detach())
    
    del model, tokenizer
    _ = gc.collect()
    libc.malloc_trim(0)
    torch.cuda.empty_cache()

    globals()[f'predictions_model{i}'] = torch.cat(globals()[f'predictions_model{i}'])
    globals()[f'predictions_model{i}'] = softmax(globals()[f'predictions_model{i}'], axis=1).numpy()

In [ ]:
predictions = None


for i, wt in enumerate(wts):
    
    if predictions is None:
        predictions = ((globals()[f'predictions_model{i}'])) * wts[i]
    else:
        predictions += ((globals()[f'predictions_model{i}'])) * wts[i]


predictions.shape

In [ ]:
time_check()

# Submission

In [ ]:
predictions_overall = []

for i in range(0, len(predictions)):
    
    predict = np.array(list("ABCDE"))[np.argsort(predictions[i])][-3:].tolist()[::-1]
    predictions_overall.append(predict)

predictions_overall = [" ".join(i) for i in predictions_overall]

In [ ]:
test_df['prediction'] = predictions_overall

In [ ]:
submission = test_df[['id', 'prediction']]
submission.to_csv('submission.csv', index=False)

pd.read_csv('submission.csv').head(10)

In [ ]:
time_check()